# Experimentos de mi proyecto: predecir las unidades vendidas por tienda

Cada modelo de la Semana 10 queda registrado en MLflow, con sus hiperparámetros, la fecha de corte, la huella de los datos, el error absoluto medio (MAE) y el pipeline entrenado. La preparación y la división se importan de `pipeline_proyecto.py`.

In [1]:
import hashlib
import logging
import time

import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from pipeline_proyecto import CORTE, RUTA, X_ent, X_prueba, preparacion, y_ent, y_prueba

logging.getLogger("mlflow").setLevel(logging.ERROR)


def huella(ruta):
    with open(ruta, "rb") as archivo:
        return hashlib.md5(archivo.read()).hexdigest()[:12]


huella_datos = huella(RUTA)
EXPERIMENTO = "pronostico-ventas-tiendas"
mlflow.set_experiment(EXPERIMENTO)
print("Registro:", mlflow.get_tracking_uri(), "| Experimento:", EXPERIMENTO, "| Huella:", huella_datos)

Filas y columnas del archivo: (468820, 15)
Entrenamiento: fechas 2011-01-29 a 2015-12-31
Prueba: fechas 2016-01-01 a 2016-05-22
Vacios en entrenamiento: {}
Vacios en prueba: {'precio_relativo': 21}


Entrenamiento: (424975, 26) Prueba: (43845, 26)
['numericas__SELL_PRICE', 'numericas__precio_relativo', 'numericas__dias_desde_lanzamiento', 'categoricas__CAT_ID_FOODS', 'categoricas__CAT_ID_HOBBIES', 'categoricas__CAT_ID_HOUSEHOLD', 'categoricas__STORE_ID_CA_1', 'categoricas__STORE_ID_CA_2', 'categoricas__STORE_ID_CA_3', 'categoricas__STORE_ID_CA_4', 'categoricas__STORE_ID_TX_1', 'categoricas__STORE_ID_TX_2', 'categoricas__STORE_ID_TX_3', 'categoricas__STORE_ID_WI_1', 'categoricas__STORE_ID_WI_2', 'categoricas__STORE_ID_WI_3', 'categoricas__WEEKDAY_Friday', 'categoricas__WEEKDAY_Monday', 'categoricas__WEEKDAY_Saturday', 'categoricas__WEEKDAY_Sunday', 'categoricas__WEEKDAY_Thursday', 'categoricas__WEEKDAY_Tuesday', 'categoricas__WEEKDAY_Wednesday', 'ya_listas__IS_WEEKEND', 'ya_listas__IS_EVENT_DAY', 'ya_listas__IS_SNAP']
Medianas que aprendio el imputador: {'SELL_PRICE': 3.44, 'precio_relativo': 1.0, 'dias_desde_lanzamiento': 729.0}
Medianas de entrenamiento: {'SELL_PRICE': 3.44, 'prec

## Entrenar y registrar

In [2]:
def evaluar(nombre, modelo):
    # preparacion viene ajustada desde pipeline_proyecto: clone la deja sin ajustar,
    # para que cada corrida aprenda solo con su entrenamiento.
    pipeline = Pipeline([("preparar", clone(preparacion)), ("modelo", modelo)])

    with mlflow.start_run(run_name=nombre):
        parametros = modelo.get_params()
        parametros["division"] = "por fecha"
        parametros["fecha_corte"] = CORTE
        parametros["huella_datos"] = huella_datos
        parametros["filas_entrenamiento"] = len(X_ent)
        mlflow.log_params(parametros)

        inicio = time.perf_counter()
        pipeline.fit(X_ent, y_ent)
        segundos = time.perf_counter() - inicio
        pred_ent = pipeline.predict(X_ent)
        pred_prueba = pipeline.predict(X_prueba)
        mae_ent = mean_absolute_error(y_ent, pred_ent)
        mae_prueba = mean_absolute_error(y_prueba, pred_prueba)
        # Metrica secundaria: castiga mas los errores grandes, los dias de muchas ventas.
        rmse_prueba = mean_squared_error(y_prueba, pred_prueba) ** 0.5

        mlflow.log_metrics({
            "mae_entrenamiento": mae_ent,
            "mae_prueba": mae_prueba,
            "rmse_prueba": rmse_prueba,
            "segundos": segundos,
        })
        mlflow.sklearn.log_model(pipeline, "modelo")

    print(f"{nombre:30s} entrenamiento {mae_ent:7.4f}   prueba {mae_prueba:7.4f}   rmse {rmse_prueba:7.4f}   {segundos:7.1f} s")

## Los modelos de la Semana 10

In [3]:
evaluar("linea base (mediana)", DummyRegressor(strategy="median"))
evaluar("ridge alpha=10", Ridge(alpha=10.0))
evaluar("bosque aleatorio", RandomForestRegressor(n_estimators=100, max_depth=12, min_samples_leaf=50, n_jobs=-1, random_state=42))
evaluar("gradient boosting (absoluto)", GradientBoostingRegressor(loss="absolute_error", random_state=42))
evaluar("red neuronal", MLPRegressor(hidden_layer_sizes=(64,), max_iter=200, random_state=42))

linea base (mediana)           entrenamiento  1.4370   prueba  1.4103   rmse  3.8576       0.6 s


ridge alpha=10                 entrenamiento  1.8098   prueba  1.7202   rmse  3.5216       0.7 s


bosque aleatorio               entrenamiento  1.5368   prueba  1.5069   rmse  3.1999      76.5 s


gradient boosting (absoluto)   entrenamiento  1.4370   prueba  1.4103   rmse  3.8576       3.1 s


red neuronal                   entrenamiento  1.7012   prueba  1.6706   rmse  3.4048     195.1 s


## El registro, ordenado

In [4]:
corridas = mlflow.search_runs(experiment_names=[EXPERIMENTO], order_by=["metrics.mae_prueba ASC"])
print(corridas[["run_id", "tags.mlflow.runName", "metrics.mae_entrenamiento", "metrics.mae_prueba", "metrics.rmse_prueba", "metrics.segundos"]].to_string(index=False))

                          run_id          tags.mlflow.runName  metrics.mae_entrenamiento  metrics.mae_prueba  metrics.rmse_prueba  metrics.segundos
2c121f233c144825b59ec881e18e5d8d gradient boosting (absoluto)                   1.437045            1.410309             3.857612          3.124587
3682bb02c83842d3a74e47311145dd70         linea base (mediana)                   1.437045            1.410309             3.857612          0.581223
853a6190c5b64f2cb5aa9333c55445c0             bosque aleatorio                   1.536824            1.506945             3.199911         76.479375
f5ae5e4bbcb2452ab9eacd2de0556093                 red neuronal                   1.701205            1.670597             3.404768        195.100190
0858746efe9d4a62b986feafca7301a6               ridge alpha=10                   1.809790            1.720190             3.521613          0.715937


## Marcar el modelo final y reproducirlo

La línea base y el gradient boosting con error absoluto tienen el menor MAE, pero los dos predicen 0 unidades en todas las filas. Por eso elijo entre los modelos que sí distinguen una venta de otra.

In [5]:
MODELO_FINAL = "bosque aleatorio"

fila_final = corridas[corridas["tags.mlflow.runName"] == MODELO_FINAL].iloc[0]
run_id_final = fila_final["run_id"]
mlflow.MlflowClient().set_tag(run_id_final, "candidato", "final")

modelo_final = mlflow.sklearn.load_model("runs:/" + run_id_final + "/modelo")
mae_reproducido = mean_absolute_error(y_prueba, modelo_final.predict(X_prueba))
print("Modelo final:", MODELO_FINAL, "| run_id:", run_id_final)
print("MAE registrado: ", fila_final["metrics.mae_prueba"])
print("MAE reproducido:", mae_reproducido)
assert abs(mae_reproducido - fila_final["metrics.mae_prueba"]) < 1e-9
print("El modelo guardado reproduce la metrica registrada.")

Modelo final: bosque aleatorio | run_id: 853a6190c5b64f2cb5aa9333c55445c0
MAE registrado:  1.5069454563287006
MAE reproducido: 1.5069454563287006
El modelo guardado reproduce la metrica registrada.


In [6]:
# Por que no me quedo con "siempre 0": el error de cada uno segun el tipo de dia.
reales = y_prueba.to_numpy()
pred_final = modelo_final.predict(X_prueba)
for nombre, filas in [("0 unidades", reales == 0), ("1 o mas unidades", reales >= 1), ("10 o mas unidades", reales >= 10)]:
    mae_cero = abs(reales[filas]).mean()
    mae_final = abs(reales[filas] - pred_final[filas]).mean()
    print(f"{nombre:18s} {filas.sum():6d} filas   MAE siempre 0: {mae_cero:7.3f}   MAE {MODELO_FINAL}: {mae_final:7.3f}")
print("Unidades reales en prueba:", int(reales.sum()), "| predichas:", round(float(pred_final.sum())))

0 unidades          24862 filas   MAE siempre 0:   0.000   MAE bosque aleatorio:   1.019
1 o mas unidades    18983 filas   MAE siempre 0:   3.257   MAE bosque aleatorio:   2.146
10 o mas unidades    1086 filas   MAE siempre 0:  18.100   MAE bosque aleatorio:  13.065
Unidades reales en prueba: 61835 | predichas: 60441


## Exportar la tabla de corridas

In [7]:
mlflow.search_runs(experiment_names=[EXPERIMENTO], order_by=["metrics.mae_prueba ASC"]).to_csv("corridas_proyecto.csv", index=False)
print("Escrito corridas_proyecto.csv")

Escrito corridas_proyecto.csv
